# Phase 1.4: analysis + figures (Colab, CPU runtime is fine)
Runs `scripts/run_analysis.py` on the full per-image outputs (Drive `per_image/`, tag `full`; D5 `full_sub`),
then `scripts/make_figures.py`. Packs ONLY aggregate JSON + chart PNGs into Drive `phase1_outputs.zip`
for the Mac. No per-image data goes into the zip.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, json, shutil, zipfile, glob
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/child-detection-audit')
REPO = pathlib.Path('/content/child-detection-skin-tone-audit')
os.environ['CHILD_AUDIT_ENV'] = 'colab'
os.chdir('/content')
!rm -rf {REPO} && git clone -q https://github.com/isaacmutuma/child-skintone-detection.git {REPO}
!git -C {REPO} log --oneline -1
os.chdir(REPO)
!pip -q install statsmodels
# run provenance JSONs (aggregate) from Drive -> repo results/phase1/runs
(REPO / 'results/phase1/runs').mkdir(parents=True, exist_ok=True)
for f in glob.glob(str(DRIVE_ROOT / 'results/phase1/runs/*__full*.json')):
    shutil.copy(f, REPO / 'results/phase1/runs')
for f in ['vlm_subsample.json', 'd5_gate_dev.json']:
    if (DRIVE_ROOT / 'results/phase1' / f).exists():
        shutil.copy(DRIVE_ROOT / 'results/phase1' / f, REPO / 'results/phase1')
!ls results/phase1/runs

In [ ]:
!python scripts/run_analysis.py --tag full
!python scripts/make_figures.py

In [ ]:
pe = json.loads((REPO / 'results/phase1/primary_endpoint.json').read_text())
for ds, d in pe['datasets'].items():
    print(f"== {ds}  H1 {d['h1_count']} supported={d['h1_supported']}  missing={d['missing_detectors']}")
    for det, v in d['detectors'].items():
        print(f"   {det:13s} gap={v['tpr_gap']:+.4f} CI=[{v['ci'][0]:+.4f}, {v['ci'][1]:+.4f}] "
              f"p_perm={v['p_perm']:.4f} p_holm={v['p_holm']:.4f} n={v['n_minor']}")
out = DRIVE_ROOT / 'phase1_outputs.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob('results/phase1/*.json')) + sorted(glob.glob('results/phase1/runs/*.json')) \
             + sorted(glob.glob('figures/phase1_*.png')):
        assert 'per_image' not in f
        z.write(f, f)
print('wrote', out, [n for n in zipfile.ZipFile(out).namelist()])